# Notebook 06: Train Baseline Models
**GlobalFlood AI: Global Multimodal Flood Prediction System (48-Hour Forecast)**

### Objective
Establish benchmark baselines to rigorously evaluate the value added by multimodal fusion:
1. **Trivial Majority Class Baseline:** Baseline probability predicting majority class frequency
2. **Tabular Logistic Regression Baseline:** Uses aggregated rainfall totals + topography (elevation, slope, TWI)
3. **Weather-Only GRU Baseline:** Ablation model ignoring SAR and terrain

### Inputs & Outputs
- **Inputs:** Train and validation samples
- **Outputs:** Trained baseline model checkpoints and comparative metrics
- **Dependencies:** `scikit-learn`, `joblib`, `ml.src.train`, `ml.src.model`


In [ ]:
import json
from pathlib import Path
from ml.src.train import train_baseline_model
from app.core.config import settings

# Load train and validation samples
with open(settings.PROCESSED_DATA_DIR / "train_samples.json") as f:
    train_samples = json.load(f)
with open(settings.PROCESSED_DATA_DIR / "val_samples.json") as f:
    val_samples = json.load(f)

print(f"Loaded Train: {len(train_samples)}, Val: {len(val_samples)}")

# Train Tabular Logistic Regression baseline
clf = train_baseline_model(train_samples, val_samples, settings.BASELINE_MODEL_FILE)
print("Trained Tabular Baseline model successfully.")


### Evaluate Weather-Only GRU Ablation Architecture
Verify that the Weather-Only GRU forward pass executes with identical sequence inputs.


In [ ]:
import torch
from ml.src.model import WeatherOnlyGRU

weather_model = WeatherOnlyGRU(feature_dim=6, hidden_dim=64)
B = 2
hist_seq = torch.randn(B, 168, 6)
hist_m = torch.ones(B, 168, 6)
fc_seq = torch.randn(B, 48, 6)
fc_m = torch.ones(B, 48, 6)

logits = weather_model(hist_seq, hist_m, fc_seq, fc_m)
print("Weather-Only GRU output shape:", logits.shape)
assert logits.shape == (B,)
print("\n[SUCCESS] Baseline models trained and ablation architectures verified.")
